# Step 3 â€” SQLite hash catalog + FASTA blob store (chr22, reference proteome)

Per the handoff's storage design: **SQLite holds metadata + hashes only, never sequence bytes twice.**
The FASTA files are the blob store; sequences are retrieved by accession (faidx-style random access),
not duplicated into the database.

**Environment note:** the handoff specifies bgzip + `samtools faidx` for the FASTA blob store. This
machine has no samtools/bgzip installed (Windows, no WSL set up yet). We use plain-text FASTA +
`pyfaidx` instead, which gives the same random-access-by-accession behavior without those binaries.
Revisit with bgzip+samtools (likely via WSL) once the pipeline moves to genome-scale / the trio
`bcftools consensus` phase, which needs samtools/bcftools regardless.

In [2]:
import os
import sqlite3
import sys

sys.path.insert(0, os.path.abspath("../scripts"))
import gencode_cds_extract as cds

REF = "../data/reference"
OUT = "../data/derived/chr22"
os.makedirs(OUT, exist_ok=True)

SOURCE = "GENCODE"
RELEASE = "v46"
CHROM = "chr22"

## Re-run the validated extraction (promoted script)

Reuses `scripts/gencode_cds_extract.py`, promoted from notebook 02 once its logic was validated
(1341/1398 chr22 transcripts, 0 unexplained exclusions).

In [3]:
catalog, flagged = cds.extract_chrom(
    os.path.join(REF, "gencode.v46.basic.annotation.gtf.gz"),
    os.path.join(REF, "gencode.v46.pc_transcripts.fa.gz"),
    os.path.join(REF, "gencode.v46.pc_translations.fa.gz"),
    CHROM,
)
print(f"validated: {len(catalog)}   flagged: {len(flagged)}")

validated: 1341   flagged: 57


## Write the FASTA blob store

Three files, one per `seq_type`. Accession is the natural ID for that layer: `ENSP...` for protein,
`ENST...` for whole CDS, `ENST....exonN` for per-exon CDS chunks.

In [4]:
proteins_fa = os.path.join(OUT, "chr22_proteins.fa")
cds_fa = os.path.join(OUT, "chr22_cds.fa")
exons_fa = os.path.join(OUT, "chr22_exons.fa")

def write_fasta(path, records):
    with open(path, "w") as fh:
        for accession, seq in records:
            fh.write(f">{accession}\n")
            for i in range(0, len(seq), 60):
                fh.write(seq[i:i+60] + "\n")

write_fasta(proteins_fa, [(e["protein_id"], e["protein_seq"]) for e in catalog])
write_fasta(cds_fa, [(e["transcript_id"], e["cds_seq"]) for e in catalog])
write_fasta(exons_fa, [
    (f"{e['transcript_id']}.exon{i+1}", seq)
    for e in catalog for i, seq in enumerate(e["exon_seqs"])
])

print("wrote", proteins_fa)
print("wrote", cds_fa)
print("wrote", exons_fa)

wrote ../data/derived/chr22/chr22_proteins.fa
wrote ../data/derived/chr22/chr22_cds.fa
wrote ../data/derived/chr22/chr22_exons.fa


In [5]:
from pyfaidx import Fasta

# building the index once here; downstream lookups just re-open with Fasta(path) and slice by key
proteins_idx = Fasta(proteins_fa)
cds_idx = Fasta(cds_fa)
exons_idx = Fasta(exons_fa)
print(f"indexed: {len(proteins_idx.keys())} proteins, {len(cds_idx.keys())} CDS, {len(exons_idx.keys())} exons")

indexed: 1341 proteins, 1341 CDS, 13494 exons


## Build the SQLite catalog

Schema from the handoff: one row per sequence, indexed on both hash columns and `gene_id`.
`low_complexity_frac` is left `NULL` here -- dustmasker/segmasker aren't installed on this machine yet;
that's the next deferred item, not silently skipped.

In [6]:
db_path = os.path.join(OUT, "hash_catalog.db")
if os.path.exists(db_path):
    os.remove(db_path)

conn = sqlite3.connect(db_path)
conn.execute("""
    CREATE TABLE sequences (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        hash_md5 TEXT NOT NULL,
        hash_sq TEXT NOT NULL,
        seq_type TEXT NOT NULL CHECK(seq_type IN ('AA','CDS','cDNA','exon')),
        accession TEXT NOT NULL,
        gene_id TEXT NOT NULL,
        source TEXT NOT NULL,
        release TEXT NOT NULL,
        evidence TEXT,
        length INTEGER NOT NULL,
        low_complexity_frac REAL
    )
""")
conn.execute("CREATE INDEX idx_hash_md5 ON sequences(hash_md5)")
conn.execute("CREATE INDEX idx_hash_sq ON sequences(hash_sq)")
conn.execute("CREATE INDEX idx_gene_id ON sequences(gene_id)")
conn.commit()
print("schema created at", db_path)

schema created at ../data/derived/chr22/hash_catalog.db


In [7]:
def evidence_string(entry):
    return f"level={entry['level']};tags={','.join(entry['tags'])}"

rows = []
for e in catalog:
    ev = evidence_string(e)
    rows.append((e["protein_md5"], e["protein_sq"], "AA", e["protein_id"], e["gene_id"],
                 SOURCE, RELEASE, ev, len(e["protein_seq"]), None))
    rows.append((e["cds_md5"], e["cds_sq"], "CDS", e["transcript_id"], e["gene_id"],
                 SOURCE, RELEASE, ev, len(e["cds_seq"]), None))
    for i, (m5, sq, n) in enumerate(e["exon_hashes"], 1):
        rows.append((m5, sq, "exon", f"{e['transcript_id']}.exon{i}", e["gene_id"],
                     SOURCE, RELEASE, ev, n, None))

conn.executemany(
    "INSERT INTO sequences (hash_md5, hash_sq, seq_type, accession, gene_id, source, release, evidence, length, low_complexity_frac) "
    "VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
    rows,
)
conn.commit()

counts = conn.execute("SELECT seq_type, COUNT(*) FROM sequences GROUP BY seq_type").fetchall()
print(f"total rows inserted: {len(rows)}")
for seq_type, n in counts:
    print(f"  {seq_type:6s}  {n}")

total rows inserted: 16176
  AA      1341
  CDS     1341
  exon    13494


## Round-trip validation

Confirms the "SQLite is metadata-only, FASTA is the blob store" design actually works: look up a hash
in SQLite, resolve it to an accession + FASTA file, pull the sequence back out via `pyfaidx`, and
re-hash it -- it must match the hash stored in SQLite, with the sequence bytes never having been
stored in the database itself.

In [8]:
FASTA_BY_TYPE = {"AA": proteins_idx, "CDS": cds_idx, "exon": exons_idx}
DIGEST_BY_TYPE_FN = {"AA": cds.md5_digest, "CDS": cds.md5_digest, "exon": cds.md5_digest}

sample = conn.execute(
    "SELECT hash_md5, hash_sq, seq_type, accession, gene_id FROM sequences "
    "WHERE seq_type = 'exon' ORDER BY RANDOM() LIMIT 1"
).fetchone()
hash_md5, hash_sq, seq_type, accession, gene_id = sample

seq = str(FASTA_BY_TYPE[seq_type][accession])
recomputed_md5 = cds.md5_digest(seq)
recomputed_sq = cds.ga4gh_sq_digest(seq)

print(f"looked up: {seq_type} {accession} (gene {gene_id})")
print(f"SQLite hash_md5:    {hash_md5}")
print(f"recomputed from FASTA: {recomputed_md5}")
assert recomputed_md5 == hash_md5
assert recomputed_sq == hash_sq
print("Round-trip OK: SQLite hash matches hash of sequence pulled from the FASTA blob store.")

looked up: exon ENST00000216264.13.exon13 (gene ENSG00000100422.14)
SQLite hash_md5:    f01c0d39090571461dfc100268768ada
recomputed from FASTA: f01c0d39090571461dfc100268768ada
Round-trip OK: SQLite hash matches hash of sequence pulled from the FASTA blob store.


## Next steps

1. Low-complexity flagging (`dustmasker`/`segmasker`) -> populate `low_complexity_frac`.
2. Bring in GIAB HG002/HG003/HG004 chr22 phased VCFs; repeat extraction per-haplotype per individual,
   landing in the same schema (same `source`/`release`, different `accession` per haplotype).
3. Compare hashes across the trio to demonstrate Mendelian inheritance (Track 1), using this catalog's
   `hash_md5`/`hash_sq` columns as the join key.
4. Note: switching to bgzip+samtools faidx for the FASTA blob store (instead of pyfaidx on plain text)
   will matter once file sizes grow past chr22 scope -- likely means setting up WSL, which is needed
   anyway for `bcftools consensus` in step 2 above.